# RSX-P1 — Paired Repair Baseline Pilot

**Exploratory GPU study, NOT a confirmatory held-out result.** Uses exactly four eligible *calibration* failures from the original protocol. Compares matched original responses across six label-blind, prefix-preserving repair scopes, with two new paired seeds per problem. Nothing is rerun from NB29H, NB29G or the historical trials. It cannot guarantee a second simultaneous Colab GPU allocation.

## Scientific controls
- **Input**: four calibration Countdown failures. **Do not use held-out failures** to choose a policy.
- **Baselines**: global regeneration, fixed 25/50/75% replacement tails, deterministic first answer-locus heuristic, randomized *prefix-cut policy* (not arbitrary position remasking).
- **Important**: official ALoDLM decoder supports prefix continuation here, not true arbitrary inpainting. Do not equate these with arbitrary-span remasking.
- **Matched controls**: same original problem, source generation and two new repair seeds across actions. Tokens/loop passes/time are recorded; token length is *not guaranteed* equal across actions.
- **Limit**: 8 new trials or 28 min/session; every result immediately saved to a dedicated Google Drive folder. Run this notebook again only to resume unfinished **P1** trial IDs.
- **Interpretation**: descriptive estimates only (four distinct failures), no claim of Q1-level statistical confirmation.

In [ ]:
import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX_P1'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Repository pinned:',CODE_COMMIT)


In [ ]:
import os,sys,time,gc,json,random,hashlib,math,re,ast,csv,zipfile,io,base64
from pathlib import Path
from types import SimpleNamespace
from collections import defaultdict,Counter
from datetime import datetime, timezone
import torch, numpy as np, pandas as pd
from huggingface_hub import snapshot_download
from google.colab import drive, files
from IPython.display import display
if not torch.cuda.is_available():raise RuntimeError('Requires GPU; Colab Runtime → T4 GPU.')
assert torch.cuda.get_device_properties(0).total_memory>=12*2**30,'Needs >=12 GiB VRAM.'
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/RepairScopeX/RSX_P1_Paired_Repair_Pilot_V1')
RECS=ROOT/'records'; RECS.mkdir(parents=True,exist_ok=True)
MODEL_ID='amazon/ALoDLM-1.7B'; MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
Q=0.4;TAU=0.4;WINDOW=16
SEEDS=[41001,41002]; ACTIONS=['global','tail25','tail50','tail75','answer_locus','random_prefix_cut']
SESSION_CAP=8;SESSION_MINS=28
PAYLOAD=base64.b64decode('eyJjYXNlcyI6W3sicHJvYmxlbSI6eyJjb2xsaXNpb25fcmV0cnkiOjAsImtpbmQiOiJjb3VudGRvd24iLCJudW1iZXJzIjpbMywxMiw4XSwicHJvYmxlbV9maW5nZXJwcmludCI6IjNlZGIzNjYwMjU2Nzc2M2JkZGI0OTkwMTA5ZGJhMDcxODNiODIyODkwODhhN2FkZTk3YzM5M2Y1OTBiNThlYWQiLCJwcm9ibGVtX2lkIjoicnN4Mjk6Y2FsaWJyYXRpb246Y291bnRkb3duOjAwMDIiLCJwcm9tcHQiOiJVc2UgdGhlIG51bWJlcnMgMywgMTIsIGFuZCA4IGV4YWN0bHkgb25jZSB0byBwcm9kdWNlIHRoZSB0YXJnZXQgMTIuIFJldHVybiBhIHZhbGlkIGFyaXRobWV0aWMgZXhwcmVzc2lvbiBhbmQgZW5kIHdpdGggYEZJTkFMX0FOU1dFUjogPG51bWJlcj5gLiIsInNwbGl0IjoiY2FsaWJyYXRpb24iLCJ0YXJnZXQiOjEyLjAsInRlbXBsYXRlIjoidHdpY2UifSwic291cmNlIjp7ImV2YWx1YXRpb24iOnsiYW5zd2VyX29yaWdpbiI6Im1pc3NpbmciLCJlbGlnaWJsZV9zZW1hbnRpY19mYWlsdXJlIjp0cnVlLCJleHBsaWNpdF9maW5hbF9udW1iZXIiOm51bGwsImZhaWx1cmVfdHlwZSI6ImRlZmluaXRlX3dyb25nIiwibnVtZXJpY19jb3JyZWN0IjpmYWxzZSwic3RyaWN0X2NvcnJlY3QiOmZhbHNlLCJzdWNjZXNzIjpmYWxzZSwidmFsaWRfZXhwcmVzc2lvbiI6bnVsbCwid3JvbmdfZXhwcmVzc2lvbiI6IjEyICsgOCAtIDMifSwiZ2VuZXJhdGVkX3Rva2VucyI6NTQsImludGVybmFsX3dhbGxfc2Vjb25kcyI6My42NTU3Nzk3Nzg5OTk5OTk1LCJraW5kIjoiY291bnRkb3duIiwibG9vcF90b2tlbl9wYXNzZXMiOjExMzYsIm1lYW5fZXhpdF9kZXB0aCI6Mi40NDQ0NDQ0NDQ0NDQ0NDQ2LCJvcmlnaW5hbF9vdXRwdXQiOiJgYGBweXRob25cbiMgVGhlIHZhbGlkIGFyaXRobWV0aWMgZXhwcmVzc2lvbiB1c2luZyAzLCAxMiwgYW5kIDggZXhhY3RseSBvbmNlIGlzIDEyICsgOCAtIDNcbnByaW50KFwiMTIgKyA4IC0gM1wiKVxucHJpbnQoXCJGSU5BTF9BTlNXRVI6IDEyXCIpXG5gYGAiLCJvcmlnaW5hbF90b2tlbl9pZHMiOls3MzU5NCwxMjY2OSwxOTgsMiw1NzYsMjY5NywzNDc4NCw3NDkzLDE2NjcsMjIwLDE4LDExLDIyMCwxNiwxNywxMSwzMjMsMjIwLDIzLDY4OTYsMzA1NSwzNzQsMjIwLDE2LDE3LDQ4OCwyMjAsMjMsNDgxLDIyMCwxOCwxOTgsMTM1MCw0NDUsMTYsMTcsNDg4LDIyMCwyMyw0ODEsMjIwLDE4LDExMzgsMTM1MCw0NDUsOTg4NDgsNjc3NTIsMzkzNTEsMjUsMjIwLDE2LDE3LDExMzgsNzM1OTRdLCJvdXRlcl9zdGVwcyI6MTUsInBlYWtfdnJhbV9nYiI6Ny44MjkwMjYyMjIyMjkwMDQsInByb2JsZW1fZmluZ2VycHJpbnQiOiIzZWRiMzY2MDI1Njc3NjNiZGRiNDk5MDEwOWRiYTA3MTgzYjgyMjg5MDg4YTdhZGU5N2MzOTNmNTkwYjU4ZWFkIiwicHJvYmxlbV9pZCI6InJzeDI5OmNhbGlicmF0aW9uOmNvdW50ZG93bjowMDAyIiwicHJvdG9jb2xfaGFzaCI6ImFmYzBjNGQwMzFiYTM5ZDU4MzcxZThjODJhNTZjYmUyNTZlMjViNTMzZmQ4ZTk0ZWQ0YzBiN2IxODQ1ZGQ3NDciLCJyZWN1cnJlbnRfcGFzc2VzIjo1OCwic3BsaXQiOiJjYWxpYnJhdGlvbiIsInN0b3BfcmVhc29uIjoiZW9zIiwidHJpYWxfaWQiOiJyc3gyOTpjYWxpYnJhdGlvbjpjb3VudGRvd246MDAwMnxuYXR1cmFsfDk5Mjk0MSIsIndhbGxfc2Vjb25kcyI6NC4xMTI3MDc1MzUwMDAwMjd9fSx7InByb2JsZW0iOnsiY29sbGlzaW9uX3JldHJ5IjowLCJraW5kIjoiY291bnRkb3duIiwibnVtYmVycyI6WzE2LDE1LDZdLCJwcm9ibGVtX2ZpbmdlcnByaW50IjoiMjNhYjYyNGUzNTQ4YzQ3NTkzOGYyNjU2YTBhZWI3OTMzMmI1YmIwMmE2MDUzYmUyZDFiNjc4OTY5YmZjZjJhYSIsInByb2JsZW1faWQiOiJyc3gyOTpjYWxpYnJhdGlvbjpjb3VudGRvd246MDAxNiIsInByb21wdCI6IlVzZSB0aGUgbnVtYmVycyAxNiwgMTUsIGFuZCA2IGV4YWN0bHkgb25jZSB0byBwcm9kdWNlIHRoZSB0YXJnZXQgMzcuIFJldHVybiBhIHZhbGlkIGFyaXRobWV0aWMgZXhwcmVzc2lvbiBhbmQgZW5kIHdpdGggYEZJTkFMX0FOU1dFUjogPG51bWJlcj5gLiIsInNwbGl0IjoiY2FsaWJyYXRpb24iLCJ0YXJnZXQiOjM3LjAsInRlbXBsYXRlIjoic3VtIn0sInNvdXJjZSI6eyJldmFsdWF0aW9uIjp7ImFuc3dlcl9vcmlnaW4iOiJtaXNzaW5nIiwiZWxpZ2libGVfc2VtYW50aWNfZmFpbHVyZSI6dHJ1ZSwiZXhwbGljaXRfZmluYWxfbnVtYmVyIjpudWxsLCJmYWlsdXJlX3R5cGUiOiJkZWZpbml0ZV93cm9uZyIsIm51bWVyaWNfY29ycmVjdCI6ZmFsc2UsInN0cmljdF9jb3JyZWN0IjpmYWxzZSwic3VjY2VzcyI6ZmFsc2UsInZhbGlkX2V4cHJlc3Npb24iOm51bGwsIndyb25nX2V4cHJlc3Npb24iOiIxNiArIDE1IC0gNiJ9LCJnZW5lcmF0ZWRfdG9rZW5zIjo2NywiaW50ZXJuYWxfd2FsbF9zZWNvbmRzIjo1LjY3ODU4OTY2MDAwMDA1Nywia2luZCI6ImNvdW50ZG93biIsImxvb3BfdG9rZW5fcGFzc2VzIjoxMzkwLCJtZWFuX2V4aXRfZGVwdGgiOjIuNDE3OTEwNDQ3NzYxMTk0LCJvcmlnaW5hbF9vdXRwdXQiOiJUbyBzb2x2ZSB0aGUgcHJvYmxlbSBvZiB1c2luZyB0aGUgbnVtYmVycyAxNiwgMTUsIGFuZCA2IGV4YWN0bHkgb25jZSB0byBwcm9kdWNlIHRoZSB0YXJnZXQgMzcsIHdlIG5lZWQgdG8gY29uc2lkZXIgdmFyaW91cyBhcml0aG1ldGljIG9wZXJhdGlvbnMgYW5kIGNvbWJpbmF0aW9ucy4gQWZ0ZXIgZXh0ZW5zaXZlIGV4cGxvcmF0aW9uLCB0aGUgdmFsaWQgZXhwcmVzc2lvbiB0aGF0IG1lZXRzIHRoZSBjcml0ZXJpYSBpczpcblxuXFxbXG5cXGJveGVkezE2ICsgMTUgLSA2fSIsIm9yaWdpbmFsX3Rva2VuX2lkcyI6WzEyNDksMTE2MjUsMjc5LDM0OTEsMzE1LDE2NjcsMjc5LDUxMDksMjIwLDE2LDIxLDExLDIyMCwxNiwyMCwxMSwzMjMsMjIwLDIxLDY4OTYsMzA1NSwzMTEsODE5MywyNzksMjE2OSwyMjAsMTgsMjIsMTEsNTgyLDExODQsMzExLDI5MDgsNTI1NywzNDc4NCw3NTI1LDMyMywyNzQ1OSwxMyw0NjM2LDE2Mzc2LDI2NDAzLDExLDI3OSwyNjk3LDc0OTMsNDI5LDIwMDI3LDI3OSwxMjg5MCwzNzQsMTQ0Nyw1OSw5NjQwLDU5LDc5MDc1LDkwLDE2LDIxLDQ4OCwyMjAsMTYsMjAsNDgxLDIyMCwyMSw5Ml0sIm91dGVyX3N0ZXBzIjoxOSwicGVha192cmFtX2diIjo3Ljg0MTg0NDA4MTg3ODY2MiwicHJvYmxlbV9maW5nZXJwcmludCI6IjIzYWI2MjRlMzU0OGM0NzU5MzhmMjY1NmEwYWViNzkzMzJiNWJiMDJhNjA1M2JlMmQxYjY3ODk2OWJmY2YyYWEiLCJwcm9ibGVtX2lkIjoicnN4Mjk6Y2FsaWJyYXRpb246Y291bnRkb3duOjAwMTYiLCJwcm90b2NvbF9oYXNoIjoiYWZjMGM0ZDAzMWJhMzlkNTgzNzFlOGM4MmE1NmNiZTI1NmUyNWI1MzNmZDhlOTRlZDRjMGI3YjE4NDVkZDc0NyIsInJlY3VycmVudF9wYXNzZXMiOjcxLCJzcGxpdCI6ImNhbGlicmF0aW9uIiwic3RvcF9yZWFzb24iOiJlb3MiLCJ0cmlhbF9pZCI6InJzeDI5OmNhbGlicmF0aW9uOmNvdW50ZG93bjowMDE2fG5hdHVyYWx8OTkyOTQxIiwid2FsbF9zZWNvbmRzIjo2LjEyNjkyNDg1NTk5OTk2fX0seyJwcm9ibGVtIjp7ImNvbGxpc2lvbl9yZXRyeSI6MCwia2luZCI6ImNvdW50ZG93biIsIm51bWJlcnMiOlsxMiwyMiwxN10sInByb2JsZW1fZmluZ2VycHJpbnQiOiI0ZTE5NGFhYjhlYmU3ODk2NTE5YmFjZGQwNWI1OWU0MmM4ZjUzODdhMDA1YmQ4MmE0ZWUxOGQzYjU2OGMyNDYzIiwicHJvYmxlbV9pZCI6InJzeDI5OmNhbGlicmF0aW9uOmNvdW50ZG93bjowMDE5IiwicHJvbXB0IjoiVXNlIHRoZSBudW1iZXJzIDEyLCAyMiwgYW5kIDE3IGV4YWN0bHkgb25jZSB0byBwcm9kdWNlIHRoZSB0YXJnZXQgMTcuIFJldHVybiBhIHZhbGlkIGFyaXRobWV0aWMgZXhwcmVzc2lvbiBhbmQgZW5kIHdpdGggYEZJTkFMX0FOU1dFUjogPG51bWJlcj5gLiIsInNwbGl0IjoiY2FsaWJyYXRpb24iLCJ0YXJnZXQiOjE3LjAsInRlbXBsYXRlIjoicGx1c21pbnVzIn0sInNvdXJjZSI6eyJldmFsdWF0aW9uIjp7ImFuc3dlcl9vcmlnaW4iOiJtaXNzaW5nIiwiZWxpZ2libGVfc2VtYW50aWNfZmFpbHVyZSI6dHJ1ZSwiZXhwbGljaXRfZmluYWxfbnVtYmVyIjpudWxsLCJmYWlsdXJlX3R5cGUiOiJkZWZpbml0ZV93cm9uZyIsIm51bWVyaWNfY29ycmVjdCI6ZmFsc2UsInN0cmljdF9jb3JyZWN0IjpmYWxzZSwic3VjY2VzcyI6ZmFsc2UsInZhbGlkX2V4cHJlc3Npb24iOm51bGwsIndyb25nX2V4cHJlc3Npb24iOiIxMiArIDE3IC0gMjIifSwiZ2VuZXJhdGVkX3Rva2VucyI6NTMsImludGVybmFsX3dhbGxfc2Vjb25kcyI6NC4zNzU3NTMzNzUwMDAwNDUsImtpbmQiOiJjb3VudGRvd24iLCJsb29wX3Rva2VuX3Bhc3NlcyI6MTExNCwibWVhbl9leGl0X2RlcHRoIjoyLjQzMzk2MjI2NDE1MDk0MzUsIm9yaWdpbmFsX291dHB1dCI6IlRvIHNvbHZlIHRoZSBwcm9ibGVtIHVzaW5nIHRoZSBudW1iZXJzIDEyLCAyMiwgYW5kIDE3IGV4YWN0bHkgb25jZSB0byBwcm9kdWNlIHRoZSB0YXJnZXQgMTcsIHdlIGNhbiB1c2UgdGhlIGZvbGxvd2luZyBhcml0aG1ldGljIGV4cHJlc3Npb246XG5cblxcW1xuXFxib3hlZHsxMiArIDE3IC0gMjJ9Iiwib3JpZ2luYWxfdG9rZW5faWRzIjpbMTI0OSwxMTYyNSwyNzksMzQ5MSwxNjY3LDI3OSw1MTA5LDIyMCwxNiwxNywxMSwyMjAsMTcsMTcsMTEsMzIzLDIyMCwxNiwyMiw2ODk2LDMwNTUsMzExLDgxOTMsMjc5LDIxNjksMjIwLDE2LDIyLDExLDU4Miw2NDYsOTkwLDI3OSwyNzAxLDM0Nzg0LDc0OTMsMTQ0Nyw1OSw5NjQwLDU5LDc5MDc1LDkwLDE2LDE3LDQ4OCwyMjAsMTYsMjIsNDgxLDIyMCwxNywxNyw5Ml0sIm91dGVyX3N0ZXBzIjoxNSwicGVha192cmFtX2diIjo3LjgzMTU4OTY5ODc5MTUwNCwicHJvYmxlbV9maW5nZXJwcmludCI6IjRlMTk0YWFiOGViZTc4OTY1MTliYWNkZDA1YjU5ZTQyYzhmNTM4N2EwMDViZDgyYTRlZTE4ZDNiNTY4YzI0NjMiLCJwcm9ibGVtX2lkIjoicnN4Mjk6Y2FsaWJyYXRpb246Y291bnRkb3duOjAwMTkiLCJwcm90b2NvbF9oYXNoIjoiYWZjMGM0ZDAzMWJhMzlkNTgzNzFlOGM4MmE1NmNiZTI1NmUyNWI1MzNmZDhlOTRlZDRjMGI3YjE4NDVkZDc0NyIsInJlY3VycmVudF9wYXNzZXMiOjU3LCJzcGxpdCI6ImNhbGlicmF0aW9uIiwic3RvcF9yZWFzb24iOiJlb3MiLCJ0cmlhbF9pZCI6InJzeDI5OmNhbGlicmF0aW9uOmNvdW50ZG93bjowMDE5fG5hdHVyYWx8OTkyOTQxIiwid2FsbF9zZWNvbmRzIjo0LjgxNDI1NjM4NDAwMDAzMn19LHsicHJvYmxlbSI6eyJjb2xsaXNpb25fcmV0cnkiOjAsImtpbmQiOiJjb3VudGRvd24iLCJudW1iZXJzIjpbMjYsMTAsMjddLCJwcm9ibGVtX2ZpbmdlcnByaW50IjoiZjA1MzZlMDMzNTkxNzIwYjllMDZmZDYyMDgwMTRmMTEyNmM2NDYyNGQ0MzUzMGJjMjRlNTM4YTM2MWU5M2IwYSIsInByb2JsZW1faWQiOiJyc3gyOTpjYWxpYnJhdGlvbjpjb3VudGRvd246MDA0MSIsInByb21wdCI6IlVzZSB0aGUgbnVtYmVycyAyNiwgMTAsIGFuZCAyNyBleGFjdGx5IG9uY2UgdG8gcHJvZHVjZSB0aGUgdGFyZ2V0IDkuIFJldHVybiBhIHZhbGlkIGFyaXRobWV0aWMgZXhwcmVzc2lvbiBhbmQgZW5kIHdpdGggYEZJTkFMX0FOU1dFUjogPG51bWJlcj5gLiIsInNwbGl0IjoiY2FsaWJyYXRpb24iLCJ0YXJnZXQiOjkuMCwidGVtcGxhdGUiOiJwbHVzbWludXMifSwic291cmNlIjp7ImV2YWx1YXRpb24iOnsiYW5zd2VyX29yaWdpbiI6Im1pc3NpbmciLCJlbGlnaWJsZV9zZW1hbnRpY19mYWlsdXJlIjp0cnVlLCJleHBsaWNpdF9maW5hbF9udW1iZXIiOm51bGwsImZhaWx1cmVfdHlwZSI6ImRlZmluaXRlX3dyb25nIiwibnVtZXJpY19jb3JyZWN0IjpmYWxzZSwic3RyaWN0X2NvcnJlY3QiOmZhbHNlLCJzdWNjZXNzIjpmYWxzZSwidmFsaWRfZXhwcmVzc2lvbiI6bnVsbCwid3JvbmdfZXhwcmVzc2lvbiI6IigyNyAtIDI2KSAqIDEwIn0sImdlbmVyYXRlZF90b2tlbnMiOjU2LCJpbnRlcm5hbF93YWxsX3NlY29uZHMiOjQuMTY0NDY0OTEzMDAwMDc0LCJraW5kIjoiY291bnRkb3duIiwibG9vcF90b2tlbl9wYXNzZXMiOjExNjAsIm1lYW5fZXhpdF9kZXB0aCI6Mi40NDY0Mjg1NzE0Mjg1NzE2LCJvcmlnaW5hbF9vdXRwdXQiOiJgYGBweXRob25cbiMgVGhlIHZhbGlkIGFyaXRobWV0aWMgZXhwcmVzc2lvbiB1c2luZyAyNiwgMTAsIGFuZCAyNyBleGFjdGx5IG9uY2UgdG8gcHJvZHVjZSA5IGlzOlxuIyAoMjcgLSAyNikgKiAxMCA9IDlcbnByaW50KFwiRklOQUxfQU5TV0VSOiA5XCIpXG5gYGAiLCJvcmlnaW5hbF90b2tlbl9pZHMiOls3MzU5NCwxMjY2OSwxOTgsMiw1NzYsMjY5NywzNDc4NCw3NDkzLDE2NjcsMjIwLDE3LDIxLDExLDIyMCwxNiwxNSwxMSwzMjMsMjIwLDE3LDIyLDY4OTYsMzA1NSwzMTEsODE5MywyMjAsMjQsMzc0LDUxMCwyLDMyMCwxNywyMiw0ODEsMjIwLDE3LDIxLDgsMzUzLDIyMCwxNiwxNSwyODQsMjIwLDI0LDE5OCwxMzUwLDQ0NSw5ODg0OCw2Nzc1MiwzOTM1MSwyNSwyMjAsMjQsMTEzOCw3MzU5NF0sIm91dGVyX3N0ZXBzIjoxNSwicGVha192cmFtX2diIjo3LjgzMTU4OTY5ODc5MTUwNCwicHJvYmxlbV9maW5nZXJwcmludCI6ImYwNTM2ZTAzMzU5MTcyMGI5ZTA2ZmQ2MjA4MDE0ZjExMjZjNjQ2MjRkNDM1MzBiYzI0ZTUzOGEzNjFlOTNiMGEiLCJwcm9ibGVtX2lkIjoicnN4Mjk6Y2FsaWJyYXRpb246Y291bnRkb3duOjAwNDEiLCJwcm90b2NvbF9oYXNoIjoiYWZjMGM0ZDAzMWJhMzlkNTgzNzFlOGM4MmE1NmNiZTI1NmUyNWI1MzNmZDhlOTRlZDRjMGI3YjE4NDVkZDc0NyIsInJlY3VycmVudF9wYXNzZXMiOjU5LCJzcGxpdCI6ImNhbGlicmF0aW9uIiwic3RvcF9yZWFzb24iOiJlb3MiLCJ0cmlhbF9pZCI6InJzeDI5OmNhbGlicmF0aW9uOmNvdW50ZG93bjowMDQxfG5hdHVyYWx8OTkyOTQxIiwid2FsbF9zZWNvbmRzIjo0LjY3NTUyMzg0MjAwMDA3NX19XSwicHJvdG9jb2xfaGFzaCI6ImFmYzBjNGQwMzFiYTM5ZDU4MzcxZThjODJhNTZjYmUyNTZlMjViNTMzZmQ4ZTk0ZWQ0YzBiN2IxODQ1ZGQ3NDcifQ==')
assert hashlib.sha256(PAYLOAD).hexdigest()=='fe6d9ec7b67aea4b8f8aaacabefdd81aafcb4b8e79a8a2cb9f1b58e89ce99f98'
input_data=json.loads(PAYLOAD)
CASES={e['problem']['problem_id']:e for e in input_data['cases']}
assert len(CASES)==4 and all(e['problem']['split']=='calibration' and e['source']['evaluation']['eligible_semantic_failure'] for e in CASES.values())
PROTOCOL={'study':'RSX_P1_PAIRED_REPAIR_EXPLORATORY_V1','parent_protocol':input_data['protocol_hash'],
 'seed_set':SEEDS,'actions':ACTIONS,'model':MODEL_ID,'model_rev':MODEL_REV,'commit':CODE_COMMIT,'q':Q,'tau':TAU,'window':WINDOW,
 'repair_decode':'entropy','input_sha256':'fe6d9ec7b67aea4b8f8aaacabefdd81aafcb4b8e79a8a2cb9f1b58e89ce99f98'}
def jhash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':')).encode()).hexdigest()
SIGNATURE=jhash(PROTOCOL)
def atomic(path,obj):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+'.tmp')
 with tmp.open('w') as f:json.dump(obj,f,sort_keys=True,indent=2);f.flush();os.fsync(f.fileno())
 os.replace(tmp,path)
manifest=ROOT/'protocol.json'
if manifest.exists():assert json.loads(manifest.read_text())=={'hash':SIGNATURE,'protocol':PROTOCOL},'Incompatible checkpoint, refusing merge.'
else:atomic(manifest,{'hash':SIGNATURE,'protocol':PROTOCOL})
def path_for(trial_id):return RECS/(hashlib.sha256(trial_id.encode()).hexdigest()+'.json')
def checked_read(trial_id):
 path=path_for(trial_id)
 if not path.exists():return None
 r=json.loads(path.read_text());assert r['trial_id']==trial_id and r['protocol_hash']==SIGNATURE
 return r
def checked_save(record):
 path=path_for(record['trial_id'])
 if path.exists():assert json.loads(path.read_text())==record,'Conflicting old trial';return
 atomic(path,record)
print('Four distinct CALIBRATION problems loaded. Held-out failures are untouched.')


In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')


In [ ]:
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
gc.collect();torch.cuda.empty_cache()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args)
model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
PROMPT_SUFFIX='\n\nSolve carefully. For Countdown, show an arithmetic expression using all given numbers exactly once. End your response with FINAL_ANSWER: <number>.'
def prompt_ids(p):
 return list(tokenizer.apply_chat_template([{'role':'user','content':p['prompt']+PROMPT_SUFFIX}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def decoded(ids):return tokenizer.decode(list(ids),skip_special_tokens=True).strip()
def set_seed(s):
 random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s)
print('Pinned ALoDLM FP32 loaded. No model generations spent on setup.')


In [ ]:
def scope_cut(ids,action,pid,seed):
 n=len(ids); assert n>=4
 if action=='global':return 0
 if action in ('tail25','tail50','tail75'):
  frac={'tail25':0.75,'tail50':0.50,'tail75':0.25}[action]
  return max(1,min(n-1,int(math.floor(n*frac))))
 if action=='random_prefix_cut':
  # Randomized policy, NOT arbitrary-span inpainting. Deterministic before inspecting repaired outcome.
  h=hashlib.sha256(f'{pid}|{seed}|RSX_P1'.encode()).digest()
  return scope_cut(ids,('tail25','tail50','tail75')[h[0]%3],pid,seed)
 if action=='answer_locus':
  rendered=decoded(ids)
  candidates=[m.start() for m in re.finditer(r'(?i)final_answer|(?<![A-Za-z])answer\s*(?:is|:)|(?:=|\bprint\s*\()',rendered)]
  if not candidates:return scope_cut(ids,'tail50',pid,seed)
  anchor=min(candidates)
  for j in range(1,n):
   if len(decoded(ids[:j]))>=anchor:return max(1,min(n-1,j-1))
  return scope_cut(ids,'tail50',pid,seed)
 raise ValueError(action)

PLAN=[(pid,a,s) for pid in sorted(CASES) for s in SEEDS for a in ACTIONS]
assert len(PLAN)==48
print('48 paired pilot repair trials; saved:',sum(checked_read(f'RSXP1|{pid}|{a}|{seed}') is not None for pid,a,seed in PLAN))


In [ ]:
started=time.monotonic();completed=0;failed=[]
for pid,action,seed in PLAN:
 if completed>=SESSION_CAP or time.monotonic()-started>=SESSION_MINS*60:break
 trial_id=f'RSXP1|{pid}|{action}|{seed}'
 if checked_read(trial_id) is not None:continue
 case=CASES[pid]; p=case['problem'];src=case['source'];ids=list(map(int,src['original_token_ids']))
 cut=scope_cut(ids,action,pid,seed);prefix=ids[:cut];remaining=len(ids)-cut
 if remaining<=0 or remaining>96:raise ValueError('Invalid repair budget')
 try:
  set_seed(seed)
  cfg=DecodeConfig(q=Q,tau=TAU,mode='entropy',window_size=WINDOW,max_new_tokens=remaining,position_penalty=None)
  cfg.validate();torch.cuda.reset_peak_memory_stats();t0=time.monotonic()
  result=decoder.generate(prompt_ids(p)+prefix,cfg)
  new_ids=list(map(int,result['token_ids']))
  repaired=decoded(prefix+new_ids)
  eval_=grade_output(repaired,p)
  entry={'trial_id':trial_id,'protocol_hash':SIGNATURE,'problem_id':pid,'split':'calibration','kind':p['kind'],
   'action':action,'seed':seed,'input_tokens':len(ids),'preserved_tokens':cut,'requested_new_tokens':remaining,
   'actual_generated_tokens':int(result.get('generated_tokens',len(new_ids))),'outer_steps':int(result.get('outer_steps',0)),
   'loop_token_passes':int(result.get('loop_token_passes',0)),'recurrent_passes':int(result.get('recurrent_passes',0)),
   'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30),'wall_seconds':round(time.monotonic()-t0,3),
   'stop_reason':str(result.get('stop_reason','unknown')),'evaluation':eval_,
   'repaired_output':repaired,'exact_prefix_invariant':(prefix+new_ids)[:cut]==prefix,'source_trial_id':src['trial_id']}
  assert entry['exact_prefix_invariant']
  checked_save(entry);completed+=1
  print('Saved',completed,action,pid,'success=',eval_['success'],flush=True)
 except torch.cuda.OutOfMemoryError:
  torch.cuda.empty_cache();failed.append({'trial_id':trial_id,'error':'CUDA OOM'});break
 except Exception as e:
  failed.append({'trial_id':trial_id,'error':repr(e)});break
if failed:atomic(ROOT/'errors_last_session.json',failed)
print('Session NEW:',completed,'All saved:',sum(checked_read(f'RSXP1|{pid}|{a}|{seed}') is not None for pid,a,seed in PLAN),'of',len(PLAN))


In [ ]:
results=[checked_read(f'RSXP1|{pid}|{a}|{seed}') for pid,a,seed in PLAN]
results=[r for r in results if r is not None]
rows=[]
for action in ACTIONS:
 subset=[r for r in results if r['action']==action]
 rows.append({'action':action,'n':len(subset),'distinct_problems':len({r['problem_id'] for r in subset}),
   'successes':sum(r['evaluation']['success'] for r in subset),
   'mean_requested_tokens':sum(r['requested_new_tokens'] for r in subset)/len(subset) if subset else None,
   'total_forward_token_passes':sum(r['loop_token_passes'] for r in subset),
   'total_wall_seconds':sum(r['wall_seconds'] for r in subset)})
summary=pd.DataFrame(rows);display(summary);summary.to_csv(ROOT/'action_summary.csv',index=False)
status={'version':PROTOCOL['study'],'planned':len(PLAN),'completed':len(results),'distinct_calibration_failures':4,
 'not_confirmatory':True,'pending_trial_ids':[f'RSXP1|{pid}|{a}|{seed}' for pid,a,seed in PLAN if checked_read(f'RSXP1|{pid}|{a}|{seed}') is None]}
atomic(ROOT/'summary.json',status)
OUT_ZIP=Path('/content/RSX_P1_PairedRepair_Checkpoint.zip')
with zipfile.ZipFile(OUT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for f in sorted(ROOT.rglob('*')):
  if f.is_file() and not f.name.endswith('.tmp'):z.write(f,f.relative_to(ROOT))
print('Export',OUT_ZIP,'completed',len(results),'/',len(PLAN))
files.download(str(OUT_ZIP))
